In [27]:
import numpy as np
import pandas as pd

In [28]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [29]:
df = pd.read_csv("covid_toy.csv")

In [30]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [31]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [32]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)
X_train

,age,gender,fever,cough,city
40,49,Female,102.0,Mild,Delhi
30,15,Male,101.0,Mild,Delhi
59,6,Female,104.0,Mild,Kolkata
32,34,Female,101.0,Strong,Delhi
83,17,Female,104.0,Mild,Kolkata
...,...,...,...,...,...
29,34,Female,NaN,Strong,Mumbai
4,65,Female,101.0,Mild,Mumbai
51,11,Female,100.0,Strong,Kolkata
57,49,Female,99.0,Strong,Bangalore


#Without ColumnTransformer (using different Encoding techniques for different data columns)

In [33]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [34]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [35]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first')
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [36]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [37]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city.toarray(),X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city.toarray(),X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

#With ColumnTransformer

In [38]:
from sklearn.compose import ColumnTransformer

In [40]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first'),['gender','city'])
],remainder='passthrough')

In [41]:
transformer.fit_transform(X_train).shape

(80, 7)

In [42]:
transformer.transform(X_test).shape

(20, 7)